In [1]:
import pandas as pd
import datetime
import sys
sys.path.append('E:\Index_Arbitrage')
import cvxpy as cp

from Framework.pythonTool import *
from Framework.riskFactor import LogSizeFactor, IndustryFactor, RiskFactor
from typing import Dict, List, Optional, Union, Tuple
from Framework.dataTool import BarraDataLoader, IndexDataLoader, PriceDataLoader, PortfolioLoader
from Framework.riskTool import PortfolioExposureCalculator
from dataclasses import dataclass

# 数据工具配置

In [5]:
dol = ddb('bctmate')
date_str = '2026.01.05'
dol.load_table_by_sql('dfs://jq_price', 'daily', f'select * from daily where date={date_str}')

,date,code,open,close,low,high,volume,money,high_limit,low_limit,avg,paused,pre_close,factor
0,2026-01-05,000001.XSHE,11.42,11.50,11.41,11.51,87549118.0,1.003479e+09,12.55,10.27,11.462,0.0,11.41,146.360309
1,2026-01-05,000002.XSHE,4.64,4.75,4.63,4.78,193853519.0,9.171435e+08,5.12,4.19,4.731,0.0,4.65,168.875961
2,2026-01-05,000004.XSHE,11.08,10.93,10.73,11.08,3785573.0,4.114829e+07,11.63,10.53,10.870,0.0,11.08,7.446000
3,2026-01-05,000006.XSHE,9.83,9.73,9.55,9.83,32218785.0,3.123836e+08,10.95,8.96,9.696,0.0,9.95,51.639522
4,2026-01-05,000007.XSHE,11.44,11.52,11.32,11.70,8273703.0,9.550195e+07,12.58,10.30,11.543,0.0,11.44,12.055000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5177,2026-01-05,688807.XSHG,189.21,196.19,187.75,197.44,3863624.0,7.475004e+08,224.76,149.84,193.471,0.0,187.30,1.000000
5178,2026-01-05,688809.XSHG,275.00,283.00,275.00,308.00,10320435.0,2.992704e+09,10000.00,0.01,289.979,0.0,256.80,1.000000
5179,2026-01-05,688819.XSHG,33.48,33.70,32.96,33.70,3234460.0,1.078420e+08,39.86,26.58,33.342,0.0,33.22,1.093523
5180,2026-01-05,688981.XSHG,126.84,129.95,126.20,130.17,91139404.0,1.174287e+10,147.40,98.26,128.845,0.0,122.83,1.000000


In [4]:
# barra相关数据获取工具
barra_data_loader = BarraDataLoader('2026-01-05')

# 指数权重获取工具
index_data_loader = IndexDataLoader('2026-01-05')

# 市场行情获取工具
price_data_loader = PriceDataLoader('2026-01-05')

# 从bctmate投资组合获取工具
portfolio_loader =  PortfolioLoader('2026-01-05')

# 风险因子配置

In [5]:
# 设置风险因子对象
factor_log_size = LogSizeFactor.from_data_loader(barra_data_loader)
factor_industry = IndustryFactor.from_data_loader(barra_data_loader)



# 获取投资组合



In [7]:
main_contracts = ['ZZ10_LS_SWAP001_L_01', 'ZZ10_LS_SWAP001_S_01']

In [26]:
main_contracts = ['HTIFS-INDEX-STOCK', 'HTIFS-INDEX-FUT']

In [27]:
main_contracts = ['GTJAINVHK-INDEX-STOCK', 'GTJAINVHK-INDEX-FUT']

In [8]:

portfolio_loader = PortfolioLoader(date='2026-01-05')
portfolio = portfolio_loader.load_portfolio_from_bctmate(main_contracts=main_contracts, portfolio_name='MyPortfolio', replace_future=True)

# 构建组合暴露计算器
portfolio_exposure_calculator = PortfolioExposureCalculator(date='2026-01-05')
portfolio_exposure_calculator.add_factor(factor_log_size)
portfolio_exposure_calculator.add_factor(factor_industry)

# 导入投资组合
portfolio_exposure_calculator.process_portfolio(portfolio)   

# 计算风险因子暴露
portfolio_exposure_calculator.calculate_factor_exposure()

print('市场风险暴露:')
print(f'多头市值: {portfolio_exposure_calculator.portfolio_exposure["market"].long_exposure["market"]}')
print(f'空头市值: {portfolio_exposure_calculator.portfolio_exposure["market"].short_exposure["market"]}')
print(f'净暴露: {portfolio_exposure_calculator.portfolio_exposure["market"].net_exposure["market"]}')
print(f'市场偏离度: {portfolio_exposure_calculator.portfolio_exposure["market"].deviation_ratio["market"]}')


print('\n各风险因子暴露:')
print('对数市值因子暴露:')
print('多头对数市值暴露:', portfolio_exposure_calculator.portfolio_exposure["log_size"].long_exposure["log_size"])
print('空头对数市值暴露:', portfolio_exposure_calculator.portfolio_exposure['log_size'].short_exposure['log_size'])
print('加权对数市值偏离度:', portfolio_exposure_calculator.portfolio_exposure['log_size'].deviation_ratio['log_size'])

print('\n行业暴露:') 
print('行业绝对偏离度加总:', portfolio_exposure_calculator.portfolio_exposure['industry'].total_abs_deviation)



市场风险暴露:
多头市值: 820702543.21
空头市值: -819925346.64
净暴露: 777196.57
市场偏离度: 0.0009478869914009858

各风险因子暴露:
对数市值因子暴露:
多头对数市值暴露: 1.020140746768658
空头对数市值暴露: -0.9944487477800033
加权对数市值偏离度: 0.026807793957635412

行业暴露:
行业绝对偏离度加总: 0.012784990000619825


In [9]:
portfolio_exposure_calculator.long_weight_breakdown

code
000001.XSHE    0.003020
000002.XSHE    0.000813
000009.XSHE    0.000819
000021.XSHE    0.001204
000027.XSHE    0.000395
                 ...   
688777.XSHG    0.001257
688778.XSHG    0.000713
688819.XSHG    0.000246
688981.XSHG    0.003705
689009.XSHG    0.001201
Name: weight, Length: 811, dtype: float64

In [10]:
portfolio_exposure_calculator.short_weight_breakdown

code
000012.XSHE   -0.00072
000019.XSHE   -0.00027
000028.XSHE   -0.00057
000029.XSHE   -0.00070
000030.XSHE   -0.00034
                ...   
603262.XSHG   -0.00014
688726.XSHG   -0.00016
301275.XSHE   -0.00021
603202.XSHG   -0.00052
603014.XSHG   -0.00020
Name: weight, Length: 1000, dtype: float64

In [33]:
index_data_loader.load_index_weight('000300.XSHG')

code
000001.XSHE    0.00438
000002.XSHE    0.00125
000063.XSHE    0.00483
000100.XSHE    0.00374
000157.XSHE    0.00170
                ...   
688506.XSHG    0.00106
688271.XSHG    0.00246
600930.XSHG    0.00073
603296.XSHG    0.00146
001391.XSHE    0.00023
Name: weight, Length: 300, dtype: float64

In [ ]:
current_weight = None
benchmark_exopsure = index_data_loader.load_index_weight('000300.XSHG')

min_weight = 0

min_weight = 0

max_weight = 1

no_buy = []

no_sell = []

buy_only = False







In [ ]:
n = 10
X = cp.Variable(n)

X >= [0,0,0,1,0,0,0,0,0,0]


NotImplementedError: Strict inequalities are not allowed.

In [27]:
factor_log_size.build_exposure_matrix(index_data_loader.load_index_weight('000300.XSHG').index)

(array([ 7.71051021,  6.33984542,  7.51190845,  6.85270673,  6.62214799,
         6.56159854,  7.25350301,  7.21251052,  6.93023989,  7.4598523 ,
         6.57248238,  7.09720371,  7.06718037,  6.65984659,  7.7333039 ,
         5.95771333,  6.70022963,  7.36682883,  6.57930607,  7.46132822,
         6.06414561,  7.40204241,  7.06601977,  8.34003946,  6.02752547,
         6.82498272,  6.58226085,  6.54483904,  6.52984955,  6.90455215,
         5.91390022,  6.17300997,  6.6490409 ,  8.27811081,  6.7015652 ,
         7.00353217,  7.06030339,  6.99555011,  7.42989057,  7.36880381,
         6.45972337,  8.90440794,  7.28904415,  8.3772105 ,  7.57807278,
         9.27618472,  6.76698575,  7.38600252,  6.20175687,  6.61009721,
         6.09322592,  7.0527893 ,  7.46975639,  7.43464614,  7.19766594,
         7.8612779 ,  6.92619836,  5.79736419,  6.55186417,  7.48399206,
         7.19215377,  6.59061158,  6.44826198,  6.33632844,  8.33967903,
         7.79185074,  7.3455623 ,  7.53916646,  6.4

In [28]:
factor_industry.build_exposure_matrix(index_data_loader.load_index_weight('000300.XSHG').index)

(array([[0., 0., 0., ..., 0., 0., 1.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 1., 0., 0.],
        ...,
        [1., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]]),
 ['交通运输I',
  '传媒I',
  '公用事业I',
  '农林牧渔I',
  '化工I',
  '医药生物I',
  '商业贸易I',
  '国防军工I',
  '家用电器I',
  '建筑材料I',
  '建筑装饰I',
  '房地产I',
  '有色金属I',
  '机械设备I',
  '汽车I',
  '煤炭I',
  '电子I',
  '电气设备I',
  '石油石化I',
  '纺织服装I',
  '美容护理I',
  '计算机I',
  '轻工制造I',
  '通信I',
  '钢铁I',
  '银行I',
  '非银金融I',
  '食品饮料I'])

In [29]:
portfolio_exposure_calculator.factors

{'log_size': <Framework.riskFactor.LogSizeFactor at 0x223540413a0>,
 'industry': <Framework.riskFactor.IndustryFactor at 0x223c00f1b20>}

In [ ]:
from pandas.core.arrays.categorical import NoNewAttributesMixin
from pandas.core.computation.ops import Op


class PortfolioOptimizer:
    """投资组合优化器"""
    
    def __init__(self, exposure_calculator: PortfolioExposureCalculator):
        """
        Args:
            exposure_calculator: 风险暴露计算器
        """
        self.exposure_calculator = exposure_calculator
        self.factors = exposure_calculator.factors
    
    def optimize_with_exposure_constraints(
        self,
        pool: List[str],
        current_weight: Optional[pd.Series] = None,
        exposure_constraints: Optional[Dict[str, Union[float, Tuple[float, float]]]] = None,
        benchmark_exposure: Optional[Dict[str, Dict]] = None,
        objective: str = 'minimize_tracking_error',
        min_weight: float = 0.0,
        max_weight: float = 1.0,
        no_buy: Optional[List[str]] = [],
        no_sell: Optional[List[str]] = [],
        buy_only: bool = True,
    ) -> Tuple[pd.Series, Dict]:
        """
        在暴露约束下优化投资组合
        
        Args:
            pool: 选股池
            target_weights: 目标权重（用于跟踪误差最小化）
            exposure_constraints: 暴露约束，{因子名: (下界, 上界)} 或 {因子名: 绝对值上限}
            benchmark_exposure: 基准暴露，{因子名: {暴露项: 值}}
            objective: 优化目标
                - 'minimize_tracking_error': 最小化跟踪误差（需要target_weights）
                - 'minimize_variance': 最小化组合方差（需要协方差矩阵，暂未实现）
                - 'maximize_expected_return': 最大化预期收益（需要预期收益，暂未实现）
            min_weight: 最小权重
            max_weight: 最大权重
            long_only: 是否只允许做多
            leverage_constraint: 杠杆约束（总权重绝对值之和的上限）
            
        Returns:
            优化后的权重Series和优化结果字典
        """
        n = len(pool)
        X = cp.Variable(n)
        
        # 构建约束
        if current_weight is None:
            current_weight = pd.Series(index=pool, data=0)

        constraints = self._build_basic_constraints(
            X, pool, current_weight, buy_only, min_weight, max_weight, no_buy, no_sell
        )

        constraints.extend(
            self._build_exposure_constraints(
                X, pool, exposure_constraints, benchmark_exposure
            )
        )
        
        # 构建目标函数
        objective_func = self._build_objective(X, pool, target_weights, objective)
        
        # 求解优化问题
        prob = self._solve_optimization(objective_func, constraints)
        
        # 构建结果
        optimized_weights, results = self._build_results(
            X, prob, pool, target_weights
        )
        
        return optimized_weights, results
    
    def _build_basic_constraints(
        self,
        X: cp.Variable,
        pool: list[str],
        current_weight,
        buy_only: bool,
        min_weight: float,
        max_weight: float,
        no_buy: list[] = [],
        no_sell: list[] = []) -> List:
        """
        构建基本约束（权重范围、权重和等）
        
        Args:
            X: 优化变量
            long_only: 是否只允许做多
            min_weight: 最小权重
            max_weight: 最大权重
            leverage_constraint: 杠杆约束
            
        Returns:
            约束列表
        """
        constraints = []
        if current_weight is None:
            current_weight_array = [0 for _ in range(len(pool))]
        else:
            current_weight_mapping = current_weight.to_dict()
            current_weight_array = [current_weight_mapping.get(code, 0) for code in pool]

        # 标的头寸绝对限制
        constraints.append(X >= min_weight)
        constraints.append(X <= max_weight)

        # 标的头寸交易限制
        buy_limit = [0 if code in no_buy else 1 for code in pool]
        sell_limit = [0 if code in no_sell else -1 for code in pool]
        constraints.append(X-current_weight_array <= buy_limit)
        constraints.append(X-current_weight_array >= sell_limit)

        # 仅买入限制
        if buy_only:
            constraints.append(X-current_weight_array >= 0)

        # 总头寸限制    
        constraints.append(cp.sum(X) == 1.0)
        
        return constraints
    
    def _build_exposure_constraints(
        self,
        X: cp.Variable,
        pool: List[str],
        current_weight: pd.Series,
        exposure_constraints: Optional[Dict[str, Union[float, Tuple[float, float]]]],
        benchmark_exposure: Optional[Dict[str, Dict]]
    ) -> List:
        """
        构建暴露约束
        
        Args:
            X: 优化变量
            pool: 选股池
            exposure_constraints: 暴露约束字典
            benchmark_exposure: 基准暴露字典
            
        Returns:
            约束列表
        """
        if not exposure_constraints:
            return []

        current_weight_array = [current_weight.get(code, 0) for code in pool]
        
        constraints = []
        
        for factor_name, factor in self.factors.items():
            if factor_name not in exposure_constraints:
                continue
            
            constraint_value = exposure_constraints[factor_name]
            exposure_matrix, exposure_items = factor.build_exposure_matrix(pool)
            
            # 计算组合暴露
            if exposure_matrix.ndim == 1:
                # 标量因子（如对数市值）
                portfolio_exposure = exposure_matrix @ X
                constraints.extend(
                    self._apply_exposure_constraint(
                        portfolio_exposure, constraint_value, factor_name,
                        benchmark_exposure, None, is_vector=False
                    )
                )
            else:
                # 向量因子（如行业）
                portfolio_exposure = exposure_matrix @ X
                constraints.extend(
                    self._apply_exposure_constraint(
                        portfolio_exposure, constraint_value, factor_name,
                        benchmark_exposure, exposure_items, is_vector=True
                    )
                )
        
        return constraints
    
    def _apply_exposure_constraint(
        self,
        portfolio_exposure: cp.Expression,
        constraint_value: Union[float, Tuple[float, float]],
        factor_name: str,
        benchmark_exposure: Optional[Dict[str, Dict]],
        exposure_items: Optional[List[str]] = None,
        is_vector: bool = False
    ) -> List:
        """
        应用暴露约束（通用方法）
        
        Args:
            portfolio_exposure: 组合暴露表达式
            constraint_value: 约束值
            factor_name: 因子名称
            benchmark_exposure: 基准暴露
            exposure_items: 暴露项列表（用于向量因子，如行业列表）
            is_vector: 是否为向量因子
            
        Returns:
            约束列表
        """
        constraints = []
        
        # 获取基准暴露
        bench_exposure = None
        if benchmark_exposure and factor_name in benchmark_exposure:
            if is_vector and exposure_items:
                # 向量因子：基准暴露是字典
                bench_exposure = np.array([
                    benchmark_exposure[factor_name].get(item, 0) 
                    for item in exposure_items
                ])
            else:
                # 标量因子：基准暴露是单个值
                bench_exposure = benchmark_exposure[factor_name].get(factor_name, 0)
        
        if isinstance(constraint_value, tuple):
            # 上下界约束
            lower, upper = constraint_value
            if bench_exposure is not None:
                constraints.append(portfolio_exposure >= bench_exposure + lower)
                constraints.append(portfolio_exposure <= bench_exposure + upper)
            else:
                constraints.append(portfolio_exposure >= lower)
                constraints.append(portfolio_exposure <= upper)
        else:
            # 绝对值约束
            if bench_exposure is not None:
                deviation = cp.abs(portfolio_exposure - bench_exposure)
                if is_vector:
                    constraints.append(cp.sum(deviation) <= constraint_value)
                else:
                    # 对于标量，支持相对偏离度
                    if abs(bench_exposure) > 1e-6:
                        constraints.append(deviation <= abs(bench_exposure) * constraint_value)
                    else:
                        constraints.append(deviation <= constraint_value)
            else:
                if is_vector:
                    constraints.append(cp.sum(cp.abs(portfolio_exposure)) <= constraint_value)
                else:
                    constraints.append(cp.abs(portfolio_exposure) <= constraint_value)
        
        return constraints
    
    def _build_objective(
        self,
        X: cp.Variable,
        pool: List[str],
        target_weights: Optional[pd.Series],
        objective: str
    ) -> cp.Minimize:
        """
        构建目标函数
        
        Args:
            X: 优化变量
            pool: 选股池
            target_weights: 目标权重
            objective: 目标函数类型
            
        Returns:
            目标函数表达式
        """
        if objective == 'minimize_tracking_error':
            if target_weights is None:
                raise ValueError("target_weights required for minimize_tracking_error objective")
            
            # 构建目标权重数组
            target_array = np.array([target_weights.get(code, 0.0) for code in pool])
            
            return cp.Minimize(cp.sum(cp.abs(X - target_array)))
        else:
            raise ValueError(f"Unknown objective: {objective}")
    
    def _solve_optimization(
        self,
        objective_func: cp.Minimize,
        constraints: List
    ) -> cp.Problem:
        """
        求解优化问题
        
        Args:
            objective_func: 目标函数
            constraints: 约束列表
            
        Returns:
            优化问题对象（已求解）
        """
        prob = cp.Problem(objective_func, constraints)
        prob.solve(solver=cp.ECOS)
        
        if prob.status != 'optimal':
            raise RuntimeError(f"Optimization failed with status: {prob.status}")
        
        return prob
    
    def _build_results(
        self,
        X: cp.Variable,
        prob: cp.Problem,
        pool: List[str],
        target_weights: Optional[pd.Series]
    ) -> Tuple[pd.Series, Dict]:
        """
        构建优化结果
        
        Args:
            X: 优化变量
            prob: 优化问题对象（已求解）
            pool: 选股池
            target_weights: 目标权重
            
        Returns:
            (优化后的权重Series, 结果字典)
        """
        # 构建权重Series
        optimized_weights = pd.Series(X.value, index=pool)
        optimized_weights = optimized_weights[abs(optimized_weights) > 1e-6]  # 去掉接近0的权重
        
        # 计算实际暴露
        actual_exposures = self.exposure_calculator.calculate_net_exposure(optimized_weights)
        
        # 计算跟踪误差
        tracking_error = None
        if target_weights is not None:
            common_codes = set(optimized_weights.index) & set(target_weights.index)
            if common_codes:
                tracking_error = sum(abs(optimized_weights.loc[list(common_codes)] - 
                                       target_weights.loc[list(common_codes)]))
        
        results = {
            'status': prob.status,
            'value': prob.value,
            'tracking_error': tracking_error,
            'exposures': actual_exposures
        }
        
        return optimized_weights, results


class BarraRiskModel:
    """Barra风险模型主类"""
    
    def __init__(
        self,
        date: Optional[str] = None,
        data_loader: Optional[BarraDataLoader] = None,
        factors: Optional[List[RiskFactor]] = None,
        industry_data: Optional[pd.Series] = None,
        market_cap_data: Optional[pd.Series] = None
    ):
        """
        Args:
            date: 计算日期，如果为None则使用上一个交易日（当data_loader和factors都为None时使用）
            data_loader: BarraDataLoader实例，用于加载数据
            factors: 风险因子列表，如果为None则使用默认因子
            industry_data: 行业数据Series（可选），直接传入数据而不使用data_loader
            market_cap_data: 市值数据Series（可选），直接传入数据而不使用data_loader
        """
        # 确定日期
        if date is None:
            date = data_loader.date if data_loader else (datetime.datetime.today() - datetime.timedelta(days=1)).strftime('%Y-%m-%d')
        self.date = date
        
        # 创建data_loader（如果需要）
        if data_loader is None and factors is None:
            data_loader = BarraDataLoader(date)
        self.data_loader = data_loader
        
        # 创建因子
        if factors is None:
            factors = self._create_default_factors(industry_data, market_cap_data)
        
        self.exposure_calculator = PortfolioExposureCalculator(factors=factors, data_loader=data_loader)
        self.optimizer = PortfolioOptimizer(self.exposure_calculator)
    
    def _create_default_factors(self, industry_data: Optional[pd.Series], market_cap_data: Optional[pd.Series]) -> List[RiskFactor]:
        """创建默认因子"""
        factors = []
        if industry_data is not None:
            factors.append(IndustryFactor(factor_data=industry_data))
        elif self.data_loader is not None:
            factors.append(IndustryFactor.from_data_loader(self.data_loader))
        
        if market_cap_data is not None:
            factors.append(LogMarketCapFactor(factor_data=market_cap_data))
        elif self.data_loader is not None:
            factors.append(LogMarketCapFactor.from_data_loader(self.data_loader))
        
        if not factors:
            raise ValueError("必须提供data_loader、factors或直接数据")
        
        return factors
    
    def calculate_exposure(
        self,
        long_weights: pd.Series,
        short_weights: Optional[pd.Series] = None
    ) -> Dict[str, ExposureResult]:
        """
        计算组合风险暴露
        
        Args:
            long_weights: 多头权重
            short_weights: 空头权重（可选）
            
        Returns:
            风险暴露结果
        """
        return self.exposure_calculator.calculate_long_short_exposure(long_weights, short_weights)
    
    def calculate_net_exposure(self, weights: pd.Series) -> Dict[str, Dict]:
        """
        计算净暴露
        
        Args:
            weights: 权重Series（可为正负）
            
        Returns:
            净暴露字典
        """
        return self.exposure_calculator.calculate_net_exposure(weights)
    
    def optimize_portfolio(
        self,
        pool: List[str],
        target_weights: Optional[pd.Series] = None,
        exposure_constraints: Optional[Dict[str, Union[float, Tuple[float, float]]]] = None,
        benchmark_exposure: Optional[Dict[str, Dict]] = None,
        **optimizer_kwargs
    ) -> Tuple[pd.Series, Dict]:
        """
        优化投资组合
        
        Args:
            pool: 选股池
            target_weights: 目标权重
            exposure_constraints: 暴露约束
            benchmark_exposure: 基准暴露
            **optimizer_kwargs: 其他优化器参数
            
        Returns:
            优化后的权重和结果
        """
        return self.optimizer.optimize_with_exposure_constraints(
            pool=pool,
            target_weights=target_weights,
            exposure_constraints=exposure_constraints,
            benchmark_exposure=benchmark_exposure,
            **optimizer_kwargs
        )
    
    def add_factor(self, factor: RiskFactor):
        """添加自定义风险因子"""
        self.exposure_calculator.add_factor(factor)
        self.optimizer.factors = self.exposure_calculator.factors
    
    def set_benchmark(self, benchmark_weights: pd.Series) -> Dict[str, Dict]:
        """
        设置基准并计算基准暴露
        
        Args:
            benchmark_weights: 基准权重
            
        Returns:
            基准暴露字典
        """
        self.benchmark_weights = benchmark_weights
        self.benchmark_exposure = self.exposure_calculator.calculate_net_exposure(benchmark_weights)
        return self.benchmark_exposure


SyntaxError: invalid syntax (1350224435.py, line 21)

In [ ]:
main_contracts = ['ZZ10_LS_SWAP001_L_01', 'ZZ10_LS_SWAP001_S_01']
portfolio_loader = PortfolioLoader(date='2026-01-05')
portfolio_loader.load_portfolio_from_bctmate(main_contracts=main_contracts, portfolio_name='MyPortfolio')


In [ ]:
barra_data_loader = BarraDataLoader('2026-01-05')
factor_log_size = LogMarketCapFactor(barra_data_loader.load_log_size_data())
factor_industry = IndustryFactor(barra_data_loader.load_industry_data())

In [ ]:
factor_industry.calculate_exposure()

In [ ]:
ddb().load_table_by_sql('dfs://jq_price', 'index_components', 'select * from index_components where index_code="000300.XSHG"')

code         weight  index_code   date      
000001.XSHE  0.438   000300.XSHG  2025-12-31    1
                                  2026-01-01    1
                                  2026-01-02    1
                                  2026-01-03    1
                                  2026-01-04    1
                                               ..
688981.XSHG  0.972   000300.XSHG  2026-01-03    1
                                  2026-01-04    1
                                  2026-01-05    1
                                  2026-01-06    1
                                  2026-01-07    1
Name: count, Length: 2400, dtype: int64

In [162]:
barra_data_loader.load_log_size_data()

code
002880.XSHE    4.081217
300449.XSHE    3.366996
301063.XSHE    3.249487
300410.XSHE    3.429927
688360.XSHG    4.024289
                 ...   
688599.XSHG    5.988809
600141.XSHG    5.940071
603067.XSHG    5.417166
688357.XSHG    3.515421
605068.XSHG    3.734801
Name: market_cap, Length: 5182, dtype: float64